In [42]:
import os
import glob
from collections import defaultdict
import numpy as np

[{'id': 0, 'name': 'football-objects', 'supercategory': 'none'},
 {'id': 1, 'name': 'ball', 'supercategory': 'football-objects'},
 {'id': 2, 'name': 'coach', 'supercategory': 'football-objects'},
 {'id': 3, 'name': 'goalkeeper', 'supercategory': 'football-objects'},
 {'id': 4, 'name': 'player', 'supercategory': 'football-objects'},
 {'id': 5, 'name': 'referee', 'supercategory': 'football-objects'}]

labels = {1: 'ball', 2: 'coach', 3: 'goalkeeper', 4: 'player', 5: 'referee'}

def process_labels_folder(labels_dir):
    list_w, list_h = [], []
    class_total_area = defaultdict(float)
    class_total_w = defaultdict(float)
    class_total_h = defaultdict(float)
    class_count = defaultdict(int)

    # Поиск всех .txt файлов в папке labels
    label_files = glob.glob(os.path.join(labels_dir, "*.txt"))

    for file_path in label_files:
        with open(file_path, 'r') as f:
            lines = f.readlines()
        for line in lines:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 5:
                continue

            try:
                cls = int(parts[0])
                w = float(parts[3])
                h = float(parts[4])
                area = w * h

                class_total_area[cls] += area
                class_total_w[cls] += w
                class_total_h[cls] += h
                list_w.append(w)
                list_h.append(h)
                class_count[cls] += 1

            except (ValueError, IndexError):
                print(f"Warning: некорректная строка в {file_path}: {line}")
                continue

    print("Статистика ширины и высота боксов по классам:")
    for cls in sorted(class_total_area.keys()):
        avg_area = class_total_area[cls] / class_count[cls]
        avg_w = class_total_w[cls] / class_count[cls]
        avg_h = class_total_h[cls] / class_count[cls]
        print(f"Класс {labels[cls]:<11} avg_w = {avg_w:.4f}, avg_h = {avg_h:.4f}, (всего боксов: {class_count[cls]})")
        
    np_list_w = np.array(list_w)
    np_list_h = np.array(list_h)

    print("\nОбщая статистика ширины и высота боксов:")
    print(f"min_w = {np_list_w.min()}, min_h = {np_list_h.min()}")
    print(f"max_w = {np_list_w.max()}, max_h = {np_list_h.max()}")
    print(f"quantile_75 = {np.quantile(np_list_h, 0.75):.4f}")
    print(f"quantile_85 = {np.quantile(np_list_h, 0.85):.4f}")
    print(f"quantile_95 = {np.quantile(np_list_h, 0.95):.4f}")
    print(f"quantile_99 = {np.quantile(np_list_h, 0.99):.4f}")

labels_dir = "D:\model_descriptor\ML_MAGA\project\coco_yolo\labels\\train"  # ← измените, если папка называется иначе или лежит в другом месте
if not os.path.isdir(labels_dir):
    print(f"Ошибка: папка '{labels_dir}' не найдена.")
    exit(1)
process_labels_folder(labels_dir)

Статистика ширины и высота боксов по классам:
Класс ball        avg_w = 0.0086, avg_h = 0.0156, (всего боксов: 591)
Класс coach       avg_w = 0.0245, avg_h = 0.0834, (всего боксов: 164)
Класс goalkeeper  avg_w = 0.0236, avg_h = 0.0785, (всего боксов: 219)
Класс player      avg_w = 0.0231, avg_h = 0.0850, (всего боксов: 9901)
Класс referee     avg_w = 0.0218, avg_h = 0.0766, (всего боксов: 1088)

Общая статистика ширины и высота боксов:
min_w = 0.00038, min_h = 0.000556
max_w = 0.087818, max_h = 0.351852
quantile_75 = 0.0955
quantile_85 = 0.1033
quantile_95 = 0.1188
quantile_99 = 0.1415


In [1]:
from yolo_model import create_yolo_model
import torch

model = create_yolo_model(nc=5)
x = torch.randn(1, 3, 640, 640)
out = model(x)